# Détecteur de fichiers inutilisés

Ce notebook a deux modes :

1. **Mode général** : repère fichiers de code + images probablement inutilisés
   (nom introuvable ailleurs dans le projet) → les déplace dans un dossier
   `_a_verifier_avant_suppression/` pour vérification.
2. **Mode images "ça dégage"** : ne regarde QUE les images, ne cherche leur
   nom QUE dans les fichiers `.css`, `.scss`, `.js`, `.jsx`, `.ts`, `.tsx`,
   `.html`. Si le nom n'apparaît nulle part → envoyée directement à la
   **corbeille système** (récupérable si besoin, mais pas de confirmation
   fichier par fichier).

⚠️ Détection heuristique : elle cherche juste le nom du fichier dans le texte
des autres fichiers. Une image chargée dynamiquement (nom construit en JS,
ex: `"logo_" + theme + ".png"`) peut être ratée. Jette un œil rapide à la
liste avant de lancer le nettoyage si tu as ce genre de cas.

## 1. Fonctions (à exécuter une fois)

In [ ]:
import os
import shutil
from pathlib import Path

# Extensions "texte" où on cherche des références, mode général
TEXT_EXTENSIONS = {
    ".py", ".js", ".jsx", ".ts", ".tsx", ".html", ".htm", ".css", ".scss",
    ".json", ".md", ".txt", ".yml", ".yaml", ".xml", ".vue", ".php",
    ".java", ".c", ".cpp", ".h", ".rb", ".go", ".rs", ".ini", ".cfg", ".toml"
}

# Extensions testées comme "potentiellement inutilisées", mode général
CANDIDATE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp", ".bmp", ".ico",
    ".py", ".js", ".jsx", ".ts", ".tsx", ".css", ".scss", ".html"
}

# Mode images : où on cherche les références
IMAGE_REF_EXTENSIONS = {".css", ".scss", ".js", ".jsx", ".ts", ".tsx", ".html", ".htm", ".vue"}
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp", ".bmp", ".ico"}

IGNORE_DIRS = {".git", "node_modules", "__pycache__", ".venv", "venv", "_a_verifier_avant_suppression"}


def collect_files(root: Path):
    all_files = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in IGNORE_DIRS]
        for fname in filenames:
            all_files.append(Path(dirpath) / fname)
    return all_files


def build_text_corpus(files, extensions):
    corpus = ""
    for f in files:
        if f.suffix.lower() in extensions:
            try:
                corpus += f.read_text(encoding="utf-8", errors="ignore").lower()
                corpus += "\n"
            except Exception:
                pass
    return corpus


def is_referenced(fname: Path, corpus: str) -> bool:
    name_full = fname.name.lower()
    name_no_ext = fname.stem.lower()
    return name_full in corpus or (len(name_no_ext) > 2 and name_no_ext in corpus)


def human_size(n):
    for unit in ["o", "Ko", "Mo", "Go"]:
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}To"


def scan_folder(root: Path):
    """Mode général : code + images, recherche dans tous les fichiers texte."""
    all_files = collect_files(root)
    corpus = build_text_corpus(all_files, TEXT_EXTENSIONS)
    candidates = [
        f for f in all_files
        if f.suffix.lower() in CANDIDATE_EXTENSIONS and not is_referenced(f, corpus)
    ]
    candidates.sort(key=lambda f: f.stat().st_size, reverse=True)
    return candidates


def scan_unused_images(root: Path):
    """Mode images : ne garde que les images absentes du css/js/html."""
    all_files = collect_files(root)
    corpus = build_text_corpus(all_files, IMAGE_REF_EXTENSIONS)
    images = [f for f in all_files if f.suffix.lower() in IMAGE_EXTENSIONS]
    unused = [f for f in images if not is_referenced(f, corpus)]
    unused.sort(key=lambda f: f.stat().st_size, reverse=True)
    return unused


def move_files(files, root: Path, trash_dir: Path):
    if not files:
        print("Rien à déplacer.")
        return
    trash_dir.mkdir(exist_ok=True)
    for f in files:
        rel = f.relative_to(root)
        dest = trash_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(f), str(dest))
        print(f"Déplacé : {rel}")
    print(f"\n{len(files)} fichier(s) déplacé(s) vers {trash_dir}")


def delete_to_system_trash(files, root: Path):
    """Envoie direct à la corbeille du système (récupérable, mais pas de confirmation par fichier)."""
    if not files:
        print("Rien à supprimer.")
        return
    for f in files:
        rel = f.relative_to(root)
        try:
            send2trash(str(f))
            print(f"Corbeille : {rel}")
        except Exception as e:
            print(f"Échec pour {rel} : {e}")
    print(f"\n{len(files)} fichier(s) envoyé(s) à la corbeille système.")

## 2. Dossier à analyser

Le notebook analyse automatiquement le dossier dans lequel il se trouve
(là où tu l'as mis, ex: la racine de ton projet). Rien à taper.

In [ ]:
ROOT = Path.cwd()
print(f"Dossier analysé : {ROOT}")

Dossier analysé : c:\Users\flori\Desktop\clikergame


## 3A. Mode "ça dégage" — juste les images

Cherche chaque image dans le css/scss/js/ts/html du projet. Si absente →
liste, puis la cellule suivante la déplace dans
`_a_verifier_avant_suppression/` pour que tu la supprimes toi-même une fois
vérifiée.

In [ ]:
unused_images = scan_unused_images(ROOT)

if not unused_images:
    print("Aucune image inutilisée trouvée.")
else:
    total_size = sum(f.stat().st_size for f in unused_images)
    print(f"{len(unused_images)} image(s) non référencée(s) dans le css/js/html :\n")
    for f in unused_images:
        rel = f.relative_to(ROOT)
        print(f"  - {rel}  ({human_size(f.stat().st_size)})")
    print(f"\nTaille totale : {human_size(total_size)}")

2 image(s) non référencée(s) dans le css/js/html :

  - assets\img\entrenement2.gif  (133.1Ko)
  - assets\img\entrenement 3.gif  (29.0Ko)

Taille totale : 162.1Ko


In [ ]:
# Déplace les images non référencées vers _a_verifier_avant_suppression/
trash_dir = ROOT / "_a_verifier_avant_suppression"
move_files(unused_images, ROOT, trash_dir)

Déplacé : assets\img\entrenement2.gif
Déplacé : assets\img\entrenement 3.gif

2 fichier(s) déplacé(s) vers c:\Users\flori\Desktop\clikergame\_a_verifier_avant_suppression


## 3B. Mode général — code + images, vérification avant suppression

Plus prudent : déplace dans `_a_verifier_avant_suppression/` au lieu de
supprimer direct. Utile si tu veux aussi checker les scripts/css orphelins.

In [ ]:
candidates = scan_folder(ROOT)

if not candidates:
    print("Aucun fichier suspect trouvé.")
else:
    total_size = sum(f.stat().st_size for f in candidates)
    print(f"{len(candidates)} fichier(s) potentiellement inutilisé(s) :\n")
    for i, f in enumerate(candidates):
        rel = f.relative_to(ROOT)
        print(f"  [{i}] {rel}  ({human_size(f.stat().st_size)})")
    print(f"\nTaille totale : {human_size(total_size)}")

1 fichier(s) potentiellement inutilisé(s) :

  [0] assets\js\a revoi.js  (11.3Ko)

Taille totale : 11.3Ko


In [ ]:
# Option A : tout déplacer
selected = candidates

# Option B (décommente et adapte) : ne garder que certains index
# selected = [candidates[i] for i in [0, 2, 5]]

trash_dir = ROOT / "_a_verifier_avant_suppression"
move_files(selected, ROOT, trash_dir)

Déplacé : assets\js\a revoi.js

1 fichier(s) déplacé(s) vers c:\Users\flori\Desktop\clikergame\_a_verifier_avant_suppression
